In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import psycopg
from dotenv import load_dotenv
import os
import mlflow
import mlflow.xgboost
import joblib


load_dotenv(override=True)

True

# EDA

In [20]:
# Connect to PostgreSQL
DB_NAME = os.getenv("DB_NAME")
DB_SCHEMA = os.getenv("DB_SCHEMA")
DB_TABLE_NAME = os.getenv("DB_TABLE_NAME")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 5432))
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_USER = os.getenv("DB_USER")

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
) as conn:

    query = f"""
        SELECT *
        FROM {DB_NAME}.{DB_SCHEMA}.{DB_TABLE_NAME};
    """
    df = pd.read_sql(query, conn)

    conn.close()

C:\Users\lixx\AppData\Local\Temp\ipykernel_22336\2189623934.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [21]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,load_date
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,2026-10-24
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2026-10-24
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,2026-10-24
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,2026-10-24
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,2026-10-24


In [22]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
load_date          0
dtype: int64

In [23]:
#one hot encoding
encoder = OrdinalEncoder()

df[['Geography']] = encoder.fit_transform(df[['Geography']])
df[['Geography']] = df[['Geography']].convert_dtypes(int)

In [24]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,load_date
0,1,15634602,Hargrave,619,0,Female,42,2,0.00,1,1,1,101348.88,1,2026-10-24
1,2,15647311,Hill,608,2,Female,41,1,83807.86,1,0,1,112542.58,0,2026-10-24
2,3,15619304,Onio,502,0,Female,42,8,159660.80,3,1,0,113931.57,1,2026-10-24
3,4,15701354,Boni,699,0,Female,39,1,0.00,2,0,0,93826.63,0,2026-10-24
4,5,15737888,Mitchell,850,2,Female,43,2,125510.82,1,1,1,79084.10,0,2026-10-24


In [25]:

columns = [
    'CreditScore',
    'Tenure',
    'Balance',
    'Age',
    'Geography',
    'NumOfProducts',
    'IsActiveMember',
    'EstimatedSalary',
    'Exited'
]

new_df = df[columns].drop_duplicates()

In [26]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Tenure           10000 non-null  int64  
 2   Balance          10000 non-null  float64
 3   Age              10000 non-null  int64  
 4   Geography        10000 non-null  Int64  
 5   NumOfProducts    10000 non-null  int64  
 6   IsActiveMember   10000 non-null  int64  
 7   EstimatedSalary  10000 non-null  float64
 8   Exited           10000 non-null  int64  
dtypes: Int64(1), float64(2), int64(6)
memory usage: 791.0 KB


In [27]:
df_features = new_df.drop('Exited', axis=1)

df_target = new_df['Exited']

In [28]:
#Check normalization of target variable
df_target.value_counts(normalize=True)

Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64

In [29]:
X_train, X_test, y_train, y_test = train_test_split(df_features, df_target, test_size=0.3, random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test) #apply using the X_train fit / standard deviation

In [30]:
joblib.dump(
    scaler,
    "models/scaler.joblib"
)

joblib.dump(
    encoder,
    "models/encoder.joblib"
)

['models/encoder.joblib']

In [31]:
print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])

Training set size: 7000
Test set size: 3000


# Create Model

In [32]:
lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    random_state=42,
    class_weight="balanced",
    verbosity=-1
)


xgb_params = {
    "n_estimators":500,
    "learning_rate":0.05,
    "max_depth":6,
    "subsample":0.8,
    "colsample_bytree":0.8,
    "objective":"binary:logistic",
    "eval_metric":"auc",
    "random_state":42,
    "n_jobs":-1
    }

xgb_model = XGBClassifier(**xgb_params)

In [33]:
lgbm_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)

In [34]:
# Evaluation
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    # report = classification_report(y_test, y_pred)
    # cm = confusion_matrix(y_test, y_pred)

    metrics = {
        "roc_auc": roc_auc_score(y_test, y_proba),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0)
    }

    evaluation =   {
        "AUC Score": auc_score
    }

    return evaluation, metrics


In [35]:
evaluate_data = []

evaluate_data.append({
    "Model": "LightGBM",
    **evaluate_model(lgbm_model, X_test, y_test)[0]
})

evaluate_data.append({
    "Model": "XGBoost",
    **evaluate_model(xgb_model, X_test, y_test)[0]
})

pd.DataFrame(evaluate_data).sort_values(by="AUC Score", ascending=False)

c:\Users\lixx\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\lixx\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[0.02558522 0.02647318 0.10078641 ... 0.0937105  0.00920699 0.29149811]
[0.02212126 0.03118041 0.05688117 ... 0.02618816 0.0076882  0.21748647]


,Model,AUC Score
1,XGBoost,0.851536
0,LightGBM,0.842776


In [36]:
xgb_model.save_model("model/xgboost_model.json")

# Feature Importance

In [38]:
# importance_xgb = pd.Series(
#     xgb_model.feature_importances_,
#     index=X_train.columns
# ).sort_values(ascending=False)

# print(importance_xgb.head(20))

In [39]:
# # MLFLOW
# # Create an experiment
# mlflow.set_experiment("MLFlow Customer Churn Experiment")

# LLM

In [44]:
from ollama import chat, ChatResponse
import ollama
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine

In [45]:
load_dotenv()
    
def read_data(table_name):
    # Connect to PostgreSQL
    DB_NAME = os.getenv("DB_NAME")
    DB_SCHEMA = os.getenv("DB_SCHEMA")
    DB_TABLE_NAME = table_name
    DB_HOST = os.getenv("DB_HOST")
    DB_PORT = int(os.getenv("DB_PORT", 5432))
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    DB_USER = os.getenv("DB_USER")

    db_engine = create_engine(
        f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )
    
    df = pd.read_sql(f"SELECT DISTINCT * FROM {DB_SCHEMA}.{DB_TABLE_NAME};", db_engine)
    db_engine.dispose()

    return df

In [46]:
predicted_data = read_data('predicted_churn')

In [47]:
predicted_data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,load_date,predicted_churn_probability,predicted_churn
0,1174,15808182,Beneventi,478,Spain,Female,36,3,92363.30,2,1,0,44912.70,0,2026-10-24,0.136266,0
1,7928,15713426,Hancock,637,Germany,Male,30,1,122185.53,1,1,0,102566.46,1,2026-10-24,0.369345,0
2,6755,15688085,Warner,627,Spain,Female,28,3,157597.61,1,0,1,34097.22,0,2026-10-24,0.180492,0
3,2207,15597901,Chidozie,609,France,Male,34,1,0.00,1,1,1,181177.90,0,2026-10-24,0.270483,0
4,6889,15727539,Schoenheimer,618,France,Female,31,4,0.00,2,1,0,29176.04,0,2026-10-24,0.060263,0


In [62]:
customer = predicted_data.head(1)
customer

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,load_date,predicted_churn_probability,predicted_churn
0,1174,15808182,Beneventi,478,Spain,Female,36,3,92363.3,2,1,0,44912.7,0,2026-10-24,0.136266,0


In [69]:
columns = [
    'Surname',
    'CreditScore',
    'Tenure',
    'Balance',
    'Age',
    'Geography',
    'NumOfProducts',
    'IsActiveMember',
    'EstimatedSalary',
    'predicted_churn_probability'
]

customer[columns].head()['Surname'].values[0]

'Beneventi'

In [72]:

customer_data = {
    "Surname": customer[columns].head()['Surname'].values[0],
    "CreditScore": customer[columns].head()['CreditScore'].values[0],
    "Tenure": customer[columns].head()['Tenure'].values[0],
    "Balance": customer[columns].head()['Balance'].values[0],
    "Age": customer[columns].head()['Age'].values[0],
    "Geography": customer[columns].head()['Geography'].values[0],
    "NumOfProducts": customer[columns].head()['NumOfProducts'].values[0],
    "IsActiveMember": customer[columns].head()['IsActiveMember'].values[0],
    "EstimatedSalary": customer[columns].head()['EstimatedSalary'].values[0],
    "predicted_churn_probability": customer[columns].head()['predicted_churn_probability'].values[0],
}

customer_data

{'Surname': 'Beneventi',
 'CreditScore': np.int64(478),
 'Tenure': np.int64(3),
 'Balance': np.float64(92363.3),
 'Age': np.int64(36),
 'Geography': 'Spain',
 'NumOfProducts': np.int64(2),
 'IsActiveMember': np.int64(0),
 'EstimatedSalary': np.float64(44912.7),
 'predicted_churn_probability': np.float64(0.13626596)}

In [77]:
prompt = f"""
Context: 
You are a senior data analyst. You have been given a dataset containing customer information and their predicted churn status. 
Your task is to analyze the dataset and provide insights on customer churn patterns, 
including any significant features that may contribute to churn, and suggest potential strategies to reduce churn based on the analysis.

Customer Data:
{customer[columns]} 

Format Output:
[Churn Analysis]
# why the customer is likely to churn
# how to prevent churn for this customer

Important Notes: do not provide any code in your response. Only provide analysis and recommendations based on the customer data provided.
"""
messages = [
    {
        "role": "user",
        "content": prompt,
    },
]

response = chat(model="llama3.2", messages=messages)
print(response.message.content)

[Churn Analysis]

Based on the provided dataset, the following insights on customer churn patterns can be identified:

**Churn Patterns:**

The customer data indicates that customers with lower credit scores, lower tenure, and lower balance are more likely to churn. This suggests that customers who are struggling financially or have been with the company for a shorter period are more prone to leaving.

**Significant Features Contributing to Churn:**

1. Low Credit Score: Customers with credit scores below 500 are more likely to churn, indicating that poor credit management is a significant factor in churn.
2. Low Tenure: Customers with tenure of less than 2 years are more likely to churn, suggesting that customers who have recently joined the company are more prone to leaving.
3. Low Balance: Customers with balance below $10,000 are more likely to churn, indicating that customers who have low account balances are more likely to leave.
4. Low Age: Customers under 25 years of age are mor